# Exp 09: Full ratio weight

Card: [exp09_full_ratio_weight.md](../../experiments/exp09_full_ratio_weight.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, CVResult, Config, CrossValidator
from core.gradcam import GradCamFigures
from core.scores import Scores
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-full-ratio-weight-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SHOW = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    N_SHOW = 4
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp09_full_ratio_weight" / "resnet18.toml", **OVERRIDES
)
BASELINE = Config.from_toml(CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.toml")
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp09_full_ratio_weight"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; exp04 keeps its color from exp07
SERIES = {"damped (exp04)": "#eb6834", "full ratio": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training


In [ ]:
results = {"damped (exp04)": CVResult.load(BASELINE)}

In [ ]:
results["full ratio"] = CrossValidator(CONFIG).run()

## Training curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for key, ax in zip(["train_map", "val_map"], axes):
    for name, r in results.items():
        ax.plot(
            np.arange(1, r.config.epochs + 1),
            r.mean_curve(key),
            color=SERIES[name],
            lw=2,
            label=name,
        )
    ax.set(title=f"{key}, mean over folds", xlabel="epoch")
axes[1].legend(loc="lower right", title="pos_weight", alignment="left")
fig.tight_layout()
figures = {f"{PREFIX}_training.png": fig}
plt.show()

## AP and Macro F1


In [ ]:
def fold_scores(result):
    """AP per class, mAP and Macro F1 (best-F1 thresholds tuned on same fold rows), one row per fold."""
    oof = result.oof
    macro_f1 = {
        fold: Scores(part[CLASSES].to_numpy(), part[[f"p_{c}" for c in CLASSES]].to_numpy()).best_f1()["f1"].mean()
        for fold, part in oof.groupby("fold")
    }
    return result.fold_ap.assign(**{"Macro F1": pd.Series(macro_f1)})


scores = pd.concat(
    {name: fold_scores(r) for name, r in results.items()}, names=["pos_weight"]
)
scores.groupby("pos_weight", sort=False).mean().round(3)

In [ ]:
# counts only if same sign on both folds
diff = (
    fold_scores(results["full ratio"]) - fold_scores(results["damped (exp04)"])
).T.add_prefix("fold ")
diff["same sign"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(1, len(VIP), figsize=(3.6 * len(VIP), 3.6), sharey=True)
for cls, ax in zip(VIP, axes):
    for name, r in results.items():
        oof = r.oof
        precision, recall, _ = precision_recall_curve(oof[cls], oof[f"p_{cls}"])
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=name,
        )
        best = r.scores.best_f1().loc[cls]
        ax.plot(
            best["recall"],
            best["precision"],
            "o",
            ms=8,
            color=SERIES[name],
            mec="#fcfcfb",
            mew=2,
        )
    ax.set(
        title=f"{cls} ({int(oof[cls].sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
axes[0].set_ylabel("precision")
axes[-1].legend(loc="lower left", title="pos_weight; dot = best F1", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    mlflow.log_table(scores.reset_index(), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)